In [1]:
#Cell 2: unzip
!unzip -q -o "plp-project-main (NEW).zip"

In [2]:
#Cell 3 — its own cell, never chained with a ! command on the same line:
%cd /content/plp-project-main

/content/plp-project-main


In [3]:
#Cell 4 — move the DB and embeddings into place:
import shutil, os
os.makedirs("data", exist_ok=True)
shutil.move("/content/plp.db", "data/plp.db")
shutil.move("/content/sentence_embeddings.npz", "data/sentence_embeddings.npz")
print("done")

done


In [4]:
#Install dependencies
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 72.2 MB/s eta 0:00:00


In [10]:
#Cell 6 — sanity-check the taxonomy loaded correctly before spending time on anything else:
import yaml
cfg = yaml.safe_load(open("config/aspects.yaml"))
print(len(cfg["aspects"]) - 1, "aspects (excl. 'other'):")
print(list(cfg["aspects"].keys()))
print("similarity_floor:", cfg["similarity_floor"])

10 aspects (excl. 'other'):
['asset_quality', 'capital_liquidity', 'dividend_policy', 'net_interest_margin', 'management_guidance', 'income_profit_results', 'wealth_business', 'investor_stock_sentiment', 'retail_banking_sentiment', 'macro_market_environment', 'other']
similarity_floor: 0.65


In [11]:
#To run GPU
targets = ["sentiment/score.py", "aspects/embed.py"]
old = '_device = "mps" if torch.backends.mps.is_available() else "cpu"'
new = ('_device = ("cuda" if torch.cuda.is_available() else '
       '"mps" if torch.backends.mps.is_available() else "cpu")')

for path in targets:
    with open(path) as f:
        content = f.read()
    if old not in content:
        print(f"pattern not found in {path} — check manually")
        continue
    with open(path, "w") as f:
        f.write(content.replace(old, new))
    print(f"patched {path}")

pattern not found in sentiment/score.py — check manually
pattern not found in aspects/embed.py — check manually


In [12]:
#Cell 7 — assign every sentence to the new taxonomy:
!python -m aspects.assign

INFO    __main__                     Ensuring all sentences are embedded...
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/plp-project-main/aspects/assign.py", line 116, in <module>
    stats = run()
  File "/content/plp-project-main/aspects/assign.py", line 76, in run
    logger.info("Embedding cache: %s", update_embeddings())
                                       ~~~~~~~~~~~~~~~~~^^
  File "/content/plp-project-main/aspects/embed_store.py", line 53, in update
    existing_ids, existing_vecs = _load_store()
                                  ~~~~~~~~~~~^^
  File "/content/plp-project-main/aspects/embed_store.py", line 38, in _load_store
    data = np.load(STORE_PATH, allow_pickle=False)
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 476, in load
    ret = NpzFile(fid, own_fid=own_fid, allow_pickle=allow_pickle,
                  pickle_kwargs=p

In [ ]:
# Cell 8 replacement — sentiment-score a random sample only (enough to
# stratify export_candidates), instead of the full corpus. Safe to re-run:
# it only scores pairs that don't already have a score, so bumping
# SAMPLE_SIZE and re-running just tops up what's missing.

from ingest.db import get_connection, init_db, now_iso
from sentiment.score import classify_sentences, MODEL_VERSION, _SCHEMA

SAMPLE_SIZE = 12000

init_db()
conn = get_connection()
conn.executescript(_SCHEMA)

rows = conn.execute(f"""
    SELECT sa.sentence_id, sa.aspect, s.text
    FROM sentence_aspects sa
    JOIN sentences s ON s.sentence_id = sa.sentence_id
    WHERE NOT EXISTS (
        SELECT 1 FROM aspect_sentiment ase
        WHERE ase.sentence_id = sa.sentence_id AND ase.aspect = sa.aspect
          AND ase.model_version = ?
    )
    ORDER BY RANDOM()
    LIMIT {SAMPLE_SIZE}
""", (MODEL_VERSION,)).fetchall()

texts = [r[2] for r in rows]
predictions = classify_sentences(texts)
ts = now_iso()
to_insert = [
    (sid, aspect, label, conf, MODEL_VERSION, ts)
    for (sid, aspect, _text), (label, conf) in zip(rows, predictions)
]
conn.executemany(
    """INSERT OR IGNORE INTO aspect_sentiment
       (sentence_id, aspect, label, confidence, model_version, scored_at)
       VALUES (?,?,?,?,?,?)""",
    to_insert,
)
conn.commit()

counts = conn.execute(
    "SELECT label, COUNT(*) FROM aspect_sentiment WHERE model_version=? GROUP BY label",
    (MODEL_VERSION,),
).fetchall()
conn.close()

print(f"scored {len(to_insert)} new pairs this run")
print("class counts so far:", counts)

In [ ]:
#Cell 8 — score sentiment on every assigned (sentence, aspect) pair
#!python -m sentiment.score

In [ ]:
#Cell 9 — export candidates.csv (1000 train/dev + 300 test by default
!python -m preprocess.export_candidates --train-dev 1000 --test 300

INFO    __main__                     ==================================================
INFO    __main__                       total_candidates_available   90027
INFO    __main__                       test_exported                300
INFO    __main__                       train_dev_exported           1000
INFO    __main__                       output                       /content/plp-project-main/data/labeled/candidates.csv
INFO    __main__                     ==================================================
INFO    __main__                     Next: open data/labeled/candidates.csv, fill in the `label` column
INFO    __main__                     (positive/neutral/negative) per docs/annotation-guideline.md.


In [ ]:
#Cell 10 — export the blind test sheet (strips FinBERT's own predicted labels so you're not anchored on them while labeling):
!python -m preprocess.export_blind_test

INFO    __main__                     wrote 300 blind test rows to /content/plp-project-main/data/labeled/test_blind.csv
INFO    __main__                     withheld columns: llm_prelabel, base_finbert_label, base_finbert_confidence
{'rows': 300, 'path': '/content/plp-project-main/data/labeled/test_blind.csv', 'withheld': ['llm_prelabel', 'base_finbert_label', 'base_finbert_confidence']}


In [ ]:
#Cell 11 — download everything back:
from google.colab import files
files.download("data/labeled/candidates.csv")
files.download("data/labeled/test_blind.csv")
files.download("data/plp.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>